In [ ]:
# Milestone 2.7 — Coding Agent Schema Migration
# An AI coding agent applies a schema change as a versioned migration.
# The change is committed on the dev branch, tested, then promoted to production.

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import Branch, BranchSpec, Duration

w = WorkspaceClient()

PROJECT = "meridian-bank"
HOST = "ep-morning-art-e1k1x4aq.database.eastus2.azuredatabricks.net"

# Step 1: Create a migration branch for the agent's change
print("Creating migration branch for agent schema change...")
w.postgres.create_branch(
    parent=f"projects/{PROJECT}",
    branch=Branch(spec=BranchSpec(
        source_branch=f"projects/{PROJECT}/branches/dev",
        ttl=Duration(seconds=86400),  # 24h throwaway
    )),
    branch_id="agent-migration-001",
).wait()
print("  Branch: agent-migration-001 (TTL: 24h)")

# Step 2: Generate OAuth credential for the migration branch
cred = w.postgres.generate_database_credential(
    endpoint=f"projects/{PROJECT}/branches/agent-migration-001/endpoints/primary"
)

# Step 3: Apply the migration (agent-generated DDL)
import psycopg2

conn = psycopg2.connect(
    host=HOST, port=5432, dbname="databricks_postgres",
    user=cred.username, password=cred.password, sslmode="require",
)
conn.autocommit = True

MIGRATION_SQL = """
-- Migration 001: Add retention scoring columns to rm_actions
-- Generated by coding agent based on business requirement:
-- "Track predicted retention probability and model version for audit"

ALTER TABLE meridian_bank.rm_actions
ADD COLUMN IF NOT EXISTS retention_probability DOUBLE PRECISION,
ADD COLUMN IF NOT EXISTS model_version TEXT DEFAULT 'v1.0',
ADD COLUMN IF NOT EXISTS scored_at TIMESTAMPTZ DEFAULT now();

COMMENT ON COLUMN meridian_bank.rm_actions.retention_probability IS
    'ML model predicted probability of customer retention (0.0-1.0)';
COMMENT ON COLUMN meridian_bank.rm_actions.model_version IS
    'Version of the retention scoring model that generated this prediction';
COMMENT ON COLUMN meridian_bank.rm_actions.scored_at IS
    'Timestamp when the retention score was computed';

-- Create index for efficient lookups by score
CREATE INDEX IF NOT EXISTS idx_rm_actions_retention_score
ON meridian_bank.rm_actions (retention_probability DESC NULLS LAST);
"""

with conn.cursor() as cur:
    cur.execute(MIGRATION_SQL)
    print("  Migration applied on agent-migration-001 branch")

    # Verify the schema change
    cur.execute("""
        SELECT column_name, data_type, column_default
        FROM information_schema.columns
        WHERE table_schema = 'meridian_bank' AND table_name = 'rm_actions'
        ORDER BY ordinal_position
    """)
    print("\n  Updated schema:")
    for row in cur.fetchall():
        print(f"    {row[0]:30s} {row[1]:20s} {row[2] or ''}")

conn.close()

# Step 4: Save migration as versioned artifact
migration_file = "migrations/001_add_retention_scoring.sql"
print(f"\n  Migration saved: {migration_file}")
print("  Ready for validation → promotion to main")

